# Notebook 2 · Linear forecasting from scratch

Companion to lectures [3](https://jiangyou2025.github.io/kun/course/03/) and [4](https://jiangyou2025.github.io/kun/course/04/).

The simplest — and most important — forecaster is the **linear model**: predict the next value as
a weighted sum of the last `p` values. Here we call **no library regressor**; everything is
hand-written with `numpy`:

1. cut the series into a **design matrix** `X` and target `y` (lagged values as features, i.e. AR(p));
2. the model is a single **matrix multiply** `y_hat = X @ w`;
3. minimise the **least-squares** error with the **normal equation** (closed form);
4. then solve the same problem with **gradient descent** and check the two agree;
5. compare against a seasonal-naive baseline.

> Requires: `numpy`, `matplotlib`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
n = 500; t = np.arange(n)
# synthetic series: trend + weekly(7) + monthly(30) + noise
y = (20 + 0.03*t + 5*np.sin(2*np.pi*t/7) + 8*np.sin(2*np.pi*t/30)
     + np.random.normal(0, 1.0, n)).astype('float64')
print('series length:', len(y))

## 1. Build the design matrix X and target y

Use the **last `p` values** as features to predict the **next** value:

$$\mathbf{x}_i = [\,y_i,\,y_{i+1},\,\dots,\,y_{i+p-1}\,], \qquad y\text{-target}_i = y_{i+p}$$

Stacking all samples gives the design matrix `X` (`N x p`) and the target vector `y` (`N`).

In [ ]:
p = 14
def make_XY(series, p):
    X, Y = [], []
    for i in range(len(series) - p):
        X.append(series[i:i+p])     # the last p values
        Y.append(series[i+p])       # the next value
    return np.array(X), np.array(Y)

X, Y = make_XY(y, p)
print('X:', X.shape, ' Y:', Y.shape)

## 2. Chronological split + standardisation

Split by time (**never shuffle**). Standardise features with the **train** mean/std — this avoids
leakage and keeps gradient descent well-conditioned.

In [ ]:
ntr = int(len(X) * 0.8)
Xtr, Ytr = X[:ntr], Y[:ntr]
Xte, Yte = X[ntr:], Y[ntr:]

mu, sd = Xtr.mean(0), Xtr.std(0)
Xtr_s = (Xtr - mu) / sd
Xte_s = (Xte - mu) / sd

# append a column of ones for the bias (intercept)
Atr = np.hstack([Xtr_s, np.ones((len(Xtr_s), 1))])
Ate = np.hstack([Xte_s, np.ones((len(Xte_s), 1))])
print('design matrix (with bias col):', Atr.shape)

## 3. The linear model is a matrix multiply

Prediction is the design matrix times the weight vector:

$$\hat{\mathbf{y}} = A\,\mathbf{w}, \qquad A \in \mathbb{R}^{N\times d},\; \mathbf{w}\in\mathbb{R}^{d},\; \hat{\mathbf{y}}\in\mathbb{R}^{N}$$

The `i`-th prediction `y_hat_i = sum_j A[i,j] * w[j]` is just the dot product of a row with the
weights. In numpy that is `A @ w`. Below we hand-write the equivalent multiply to confirm it
matches `@`.

In [ ]:
def matmul_vec(A, w):
    """Hand-written matrix x vector: a dot product per row. Equivalent to A @ w."""
    out = np.zeros(A.shape[0])
    for i in range(A.shape[0]):
        out[i] = np.sum(A[i, :] * w)   # dot product of row i with w
    return out

w_test = np.random.randn(Atr.shape[1])
print('max |hand-written - A@w|:', np.max(np.abs(matmul_vec(Atr, w_test) - Atr @ w_test)))

## 4. Least squares + the normal equation (closed form)

We minimise the **mean squared error**:

$$\mathcal{L}(\mathbf{w}) = \frac{1}{N}\lVert A\mathbf{w} - \mathbf{y}\rVert^2$$

Setting the gradient to zero gives the **normal equation**, which has a closed-form solution:

$$A^{\top}A\,\mathbf{w} = A^{\top}\mathbf{y} \quad\Longrightarrow\quad \mathbf{w}^\star = (A^{\top}A)^{-1}A^{\top}\mathbf{y}$$

Numerically we solve the linear system with `np.linalg.solve` (more stable than inverting).

In [ ]:
w_ls = np.linalg.solve(Atr.T @ Atr, Atr.T @ Ytr)   # solve the normal equation
pred_ls = Ate @ w_ls

def rmse(a, f): return np.sqrt(np.mean((a - f)**2))
def mae(a, f):  return np.mean(np.abs(a - f))

print(f'closed-form   test  RMSE {rmse(Yte, pred_ls):.4f}   MAE {mae(Yte, pred_ls):.4f}')

## 5. Gradient descent

The normal equation is one shot, but forming `A^T A` gets expensive when `A` is large.
**Gradient descent** instead nudges the weights down the error slope. The gradient of the MSE is:

$$\nabla_{\mathbf{w}}\mathcal{L} = \frac{2}{N}A^{\top}(A\mathbf{w} - \mathbf{y})$$

Update rule: `w <- w - lr * grad`.

In [ ]:
w = np.zeros(Atr.shape[1])
lr = 0.05
N  = len(Atr)
loss_hist = []
for it in range(2000):
    err  = Atr @ w - Ytr                 # residual (N,)
    grad = (2 / N) * (Atr.T @ err)       # gradient (d,)
    w   -= lr * grad                     # update
    loss_hist.append(np.mean(err**2))

pred_gd = Ate @ w
print(f'grad-descent  test  RMSE {rmse(Yte, pred_gd):.4f}   MAE {mae(Yte, pred_gd):.4f}')
print(f'weight gap vs closed form  ||w_ls - w_gd|| = {np.linalg.norm(w_ls - w):.5f}')

plt.figure(figsize=(8, 4))
plt.plot(loss_hist)
plt.xlabel('iteration'); plt.ylabel('train MSE'); plt.yscale('log')
plt.title('Gradient-descent loss curve'); plt.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

Gradient descent converged to **almost the same weights** as the closed-form solution — two
roads to the same minimum.

## 6. Evaluate against a seasonal-naive baseline

Any model must first beat the naive baseline. Here `p=14 > 7`, so the features already contain
"the same day last week"; seasonal-naive simply takes the value from 7 steps ago.

In [ ]:
snaive = Xte[:, -7]                    # value 7 steps ago (7th-from-last feature column)
print(f'seasonal-naive  test RMSE {rmse(Yte, snaive):.4f}')
print(f'linear model    test RMSE {rmse(Yte, pred_ls):.4f}')

plt.figure(figsize=(11, 4))
plt.plot(Yte, color='black', label='actual')
plt.plot(pred_ls, '--', label='linear (closed form)')
plt.plot(snaive, ':', label='seasonal-naive')
plt.legend(); plt.title('Test set: forecast vs actual')
plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## Summary

- A linear forecaster is **design matrix x weights** — one matrix multiply is the whole forward pass.
- The **normal equation** gives the closed-form optimum; **gradient descent** iterates to the same
  solution — they agree.
- Always compare against a **baseline**; the linear model clearly beats seasonal-naive on this
  strongly-seasonal series.
- Next: [Notebook 3](03_insect_trajectory.ipynb) — take linear / neural models to a 2-D insect trajectory.
